In [ ]:
# Import DESeq2
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats
import decoupler as dc
import sc_toolbox
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

In [ ]:
def add_features_column(pw_df, msigdb_all, feature_col, results_df):
    """
    Annotate a pathway results DataFrame with direction-matched genes.

    For each pathway, finds genes that (1) belong to the pathway, (2) are
    present in `results_df`, and (3) have a ``stat`` sign matching the
    pathway activity score.

    Parameters
    ----------
    pw_df : pd.DataFrame
        Pathway activity DataFrame with columns ``variable`` (pathway name)
        and ``score``.
    msigdb_all : pd.DataFrame
        Gene-set membership table with columns ``source`` (pathway) and
        ``target`` (gene).
    feature_col : str
        Name of the new column to add to `pw_df`.
    results_df : pd.DataFrame
        DEG results with a ``stat`` column indexed by gene name.

    Returns
    -------
    pd.DataFrame
        Copy of `pw_df` with `feature_col` added, containing lists of
        direction-matched genes per pathway.
    """
    pw_df_copy = pw_df.copy()
    pw_df_copy[feature_col] = [[] for _ in range(len(pw_df_copy))]

    pathway_to_genes = msigdb_all.groupby("source")["target"].apply(list).to_dict()
    gene_to_stat = results_df["stat"].to_dict()

    for idx, row in pw_df_copy.iterrows():
        genes_in_pathway = pathway_to_genes.get(row["variable"], [])
        score = row["score"]
        matching_genes = [
            gene
            for gene in genes_in_pathway
            if gene in gene_to_stat
            and (
                (score > 0 and gene_to_stat[gene] > 0)
                or (score < 0 and gene_to_stat[gene] < 0)
            )
        ]
        pw_df_copy.at[idx, feature_col] = matching_genes

    return pw_df_copy


def pathway_analysis(
    dge_df,
    pws=None,
):
    """
    Run pathway activity scoring on DEG statistics using decoupler.
    Scores pathway activity via univariate linear model (ULM), and 
    writes the results back to CSV.
    Parameters
    ----------
    dge_df : pd.DataFrame 
        PyDEseq2 output. Must be provided.
        Expected columns: ``padj``, ``log2FoldChange``, ``stat``.
    pws : pd.DataFrame or None
        Pathway–gene network for decoupler (e.g. MSigDB). Must be provided.
        Expected columns: ``source``, ``target``, ``collection``.
    Returns
    -------
    pd.DataFrame
        ``pw_df`` — scored pathway activity DataFrames.
    """
    if pws is None or dge_df is None:
        raise ValueError("`pws` and 'dge_df' must be provided")
    dge_df = dge_df.dropna(subset=["log2FoldChange", "stat", "padj"])
    dge_df_filter_pv = dge_df[dge_df['padj'] < 0.05]
    dge_df_filter_fc15 = dge_df_filter_pv[
        (dge_df_filter_pv['log2FoldChange'] > 0.58496250072) | # 1.5 fold change threshold
        (dge_df_filter_pv['log2FoldChange'] < -0.58496250072)
    ]
    dge_df_filter_fc20 = dge_df_filter_pv[
        (dge_df_filter_pv['log2FoldChange'] > 1) |
        (dge_df_filter_pv['log2FoldChange'] < -1)
    ]
    # --- Pathway activity ---
    data = dge_df[["stat"]].T
    pw_acts, pw_padj = dc.mt.ulm(data=data, net=pws)
    pw_df = pw_acts.melt(value_name="score").merge(
        pw_padj.melt(value_name="pvalue")
        .assign(logpval=lambda x: -np.log10(x["pvalue"].clip(2.22e-4, 1)))
    )
    pw_df = pw_df.merge(
        pws[["collection", "source"]].drop_duplicates(subset="source"),
        left_on="variable",
        right_on="source",
        how="left",
    ).drop(columns="source")
    pw_df = add_features_column(pw_df, pws, "features with 1.5 FC", dge_df_filter_fc15)
    pw_df = add_features_column(pw_df, pws, "features with 2 FC", dge_df_filter_fc20)

    return pw_df

In [ ]:
import session_info2; session_info2.session_info(dependencies=True)

In [ ]:
file_path = 'tables'

In [ ]:
# read pathway info
MSigDB = pd.read_csv('tables/20251014_MSigDB.csv', index_col=0)
MSigDB = MSigDB[MSigDB['collection'].isin(['hallmark', 'kegg_pathways'])]

In [ ]:
# read count table and filter genes
counts_df = pd.read_csv('tables/count.csv', index_col=0)
genes_to_keep = counts_df.columns[counts_df.sum(axis=0) >= 10]
counts_df = counts_df[genes_to_keep]

# create meta data
metadata = pd.DataFrame({
    "condition": (["RC"] * 4 +
                  ["RC_DFMO"] * 4 +
                  ["B6"] * 4)
}, index=["RC105", "RC139", "RC140", "RC143",
          "D_RC189", "D_RC190", "D_RC181", "D_RC182",
          "B6_2", "B6_3", "B6_4", "B6_6"])

metadata.index.name = "sample"

In [ ]:
# Differential expression with PyDEseq2
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~condition",
    refit_cooks=True,
    inference=inference,
)
dds.deseq2()

In [ ]:
# contrasting RC vs B6 group
stat_res = DeseqStats(dds, contrast=["condition", "RC", "B6"], inference=inference)
stat_res.summary()
stat_res.lfc_shrink(coeff="condition[T.RC]")
results_df_rcvb6 = stat_res.results_df
results_df_rcvb6.to_csv('tables/RCvB6_DGE.csv')
pathways_rcvb6 = pathway_analysis(dge_df=results_df_rcvb6, pws=MSigDB)
pathways_rcvb6.to_csv('tables/RCvB6_DGE_PW.csv')

In [ ]:
# contrasting DMFO vs RC group
counts_df_rc = counts_df.loc[['RC105', 'RC139', 'RC140', 'RC143', 'D_RC189', 'D_RC190', 'D_RC181','D_RC182']]
metadata_rc = metadata.loc[['RC105', 'RC139', 'RC140', 'RC143', 'D_RC189', 'D_RC190', 'D_RC181','D_RC182']]
inference = DefaultInference(n_cpus=8)
dds_RC = DeseqDataSet(
    counts=counts_df_rc,
    metadata=metadata_rc,
    design="~condition",
    refit_cooks=True,
    inference=inference,
)

dds_RC.deseq2()
stat_res = DeseqStats(dds_RC, contrast=["condition", "RC_DFMO", "RC"], inference=inference)
stat_res.summary()
stat_res.lfc_shrink(coeff="condition[T.RC_DFMO]")
results_df_dmfovrc = stat_res.results_df
results_df_dmfovrc.to_csv('tables/DFMOvRC_DGE.csv')
pathways_dmfovrc = pathway_analysis(dge_df=results_df_dmfovrc, pws=MSigDB)
pathways_dmfovrc.to_csv('tables/DFMOvRC_DGE_PW.csv')

In [ ]:
# exporting data for log2FC graph in R
rescue_genelist = list(set_up_in_rc.intersection(set_down_in_dfmo)) + list(set_down_in_rc.intersection(set_up_in_dfmo))
# Create the merged Log2FC table
df_rcvb6_export = results_df_rcvb6[['log2FoldChange', 'padj']].rename(columns={
    'log2FoldChange': 'log2FC_RC_vs_B6',
    'padj':           'padj_RC_vs_B6'
})

df_dfmovrc_export = results_df_dmfovrc[['log2FoldChange', 'padj']].rename(columns={
    'log2FoldChange': 'log2FC_DFMO_vs_RC',
    'padj':           'padj_DFMO_vs_RC'
})

df_all = df_rcvb6_export.join(df_dfmovrc_export, how='outer')

# keep rows where at least one padj < 0.05 
df_all_sig = df_all[(df_all['padj_RC_vs_B6'] < 0.05) | (df_all['padj_DFMO_vs_RC'] < 0.05)]

df_all_sig['is_rescue'] = df_all_sig.index.isin(rescue_genelist)
df_all_sig.index.name = 'gene'
df_all_sig.to_csv('tables/rescue_genes_merged_log2fc.csv')